# Oracle Atlas example: source evidence to a checked economic episode

This is a **bounded working example** for first-time authors. It reproduces one real UMA request from five transactions, using the student code at immutable commit `5ee23c5ee76394233b26c89c1ebd9acae7328aa0`. It does not reproduce the full eight-protocol resource.

## Part I - Overview and navigation

An **oracle** supplies information to a blockchain program. A **transaction** is an action recorded on a chain; a **receipt** records execution and emitted events. A **record** is a row in a particular table, not necessarily a person, decision or transaction. A **principal** is a participant's own money committed to the mechanism; a **reward** is an incremental gain. Protocol adjudication does not establish independent factual truth. A blockchain address is not a verified person or institution.

Follow Parts I-VIII in order: overview, sources, querying, raw inspection, processing, processed inspection, analysis, technical validation. In Colab choose **Runtime -> Run all** on a fresh CPU runtime. A Google account is needed to obtain a hosted runtime; no wallet, API credential, GPU or paid endpoint is needed for bundled replay. Save a copy through File -> Save a copy in Drive, or download the final notebook and evidence ZIP before the session ends.

Setup installs `uv==0.12.19` in a temporary folder and creates an isolated Python 3.12 environment. It avoids assuming the notebook host has `ensurepip`. If the host uses a different Python minor version, uv obtains a compatible interpreter. The scientific dependency versions remain pinned and the kernel's packages are not upgraded. Internet is required for setup; the data stages then replay bundled real inputs offline. Source hash failures stop execution.

Expected result: 5 source transactions, 9 decoded events, 8 transfers, one 85-field episode and 28 passing case checks. **1,000 USDC.e gross = 750 returned principal + 250 token gain.** Installation and network performance affect setup time; approximately 2 GB RAM is sufficient for the case according to the student bundle. This is not a measured full-run resource budget.

The exact scientific implementation is linked from each section. The companion `examples/oracle_review/README.md` explains the metadata repair and remaining student tasks. No hosted Colab success is claimed until an executed notebook and runtime evidence have been saved.

### Setup

The next cell prints the host Python, scientific Python, exact code commit, data version, setup tool and configuration hash. On failure, keep the full diagnostic, restart the session and run from the top; do not change hashes or expected outcomes to force a pass.


In [ ]:
from IPython.display import display, Markdown
import csv
"""Set up the immutable one-case tutorial without relying on host ensurepip.

Only standard-library imports are needed in the notebook kernel. Scientific
dependencies go into a separate Python 3.12 environment. This is setup and
case replay, not reproduction of the complete cross-protocol data resource.
"""

import hashlib
import json
import os
import platform
import shutil
import subprocess
import sys
import tempfile
import time
from pathlib import Path

REPO_URL = "https://github.com/virusLuke3/Oracle4SD-Atlas.git"
CODE_REF = "5ee23c5ee76394233b26c89c1ebd9acae7328aa0"
DATA_VERSION = "uma-public-rpc-case-v1"
UV_VERSION = "0.12.19"


def command(args, cwd=None, extra_env=None, timeout=300):
    env = os.environ.copy()
    env.update({"MPLBACKEND": "Agg", "UV_NO_PROGRESS": "1"})
    env.update(extra_env or {})
    result = subprocess.run(
        [str(a) for a in args], cwd=cwd, env=env, text=True,
        stdout=subprocess.PIPE, stderr=subprocess.PIPE, timeout=timeout,
    )
    if result.returncode:
        raise RuntimeError(
            f"Setup/stage failed (exit {result.returncode}): {args!r}\n"
            f"stdout:\n{result.stdout[-6000:]}\nstderr:\n{result.stderr[-6000:]}\n"
            "Keep this diagnostic. Do not continue with partly generated outputs."
        )
    return result.stdout


def scientific_python_selector(version_info=None, executable=None):
    """Use a compatible host, otherwise ask uv for managed Python 3.12."""
    version_info = version_info or sys.version_info
    executable = executable or sys.executable
    return executable if tuple(version_info[:2]) == (3, 12) else "3.12"


def bootstrap():
    started = time.monotonic()
    work = Path(tempfile.mkdtemp(prefix="oracle-atlas-example-"))
    repo = work / "repository"
    if shutil.which("git") is None:
        raise RuntimeError("Git is required. Colab provides it; install Git for a local run.")
    command(["git", "clone", "--quiet", "--no-checkout", REPO_URL, repo])
    command(["git", "checkout", "--quiet", "--detach", CODE_REF], cwd=repo)
    actual = command(["git", "rev-parse", "HEAD"], cwd=repo).strip()
    if actual != CODE_REF:
        raise RuntimeError(f"Code identity mismatch: expected {CODE_REF}, got {actual}")

    # Installing into a temporary target does not upgrade the notebook kernel.
    # uv creates/seeds an environment without calling the host's ensurepip.
    tools_dir = work / "setup-tools"
    command([
        sys.executable, "-m", "pip", "install", "--disable-pip-version-check",
        "--target", tools_dir, f"uv=={UV_VERSION}",
    ])
    uv_env = {"PYTHONPATH": str(tools_dir)}
    uv = [sys.executable, "-m", "uv"]
    env = work / "scientific-environment"
    selector = scientific_python_selector()
    # If the host changed to another Python minor version, uv installs a
    # managed compatible interpreter. No sudo or system-package edits are used.
    command(uv + ["venv", "--python", selector, "--seed", env], extra_env=uv_env)
    python = env / ("Scripts/python.exe" if os.name == "nt" else "bin/python")
    version = command([python, "-c", "import sys; print('.'.join(map(str,sys.version_info[:2])))"]).strip()
    if version != "3.12":
        raise RuntimeError(f"Scientific environment must be Python 3.12; got {version}")
    print(command([
        python, "-m", "pip", "install", "--disable-pip-version-check",
        "-r", repo / "requirements.txt",
    ])[-1200:])
    command([python, "-m", "pip", "check"])
    command([python, "-c", "import pandas, pyarrow, eth_abi, eth_utils, matplotlib, jsonschema"])
    run = repo / "runs/tutorial-example"
    run.mkdir(parents=True)
    cfg_bytes = (repo / "configs/project.json").read_bytes()
    identity = {
        "host_python": platform.python_version(),
        "scientific_python_minor": version,
        "python_selector": selector,
        "setup_tool": f"uv=={UV_VERSION}",
        "code_commit": actual,
        "data_version": DATA_VERSION,
        "configuration_sha256": hashlib.sha256(cfg_bytes).hexdigest(),
        "setup_seconds": round(time.monotonic() - started, 3),
        "scope": "one real UMA case; not the full candidate dataset",
    }
    (work / "setup_identity.json").write_text(json.dumps(identity, indent=2) + "\n")
    print(json.dumps(identity, indent=2))
    return {"WORK": work, "REPO": repo, "PYTHON": python, "RUN": run, "ENV": env}


if __name__ == "__main__":
    globals().update(bootstrap())


## 2. Data sources and metadata

The fixed source specification names four Polygon transactions and one Ethereum transaction. A **block** is an ordered collection of transactions; a block hash identifies its exact contents. The source register lists external origins, licenses and file hashes. **SHA256** detects changed file bytes; it cannot prove a claim is factually true.

Read [metadata/source_register.csv](https://github.com/virusLuke3/Oracle4SD-Atlas/blob/5ee23c5ee76394233b26c89c1ebd9acae7328aa0/metadata/source_register.csv), [metadata/acquisition_profiles.json](https://github.com/virusLuke3/Oracle4SD-Atlas/blob/5ee23c5ee76394233b26c89c1ebd9acae7328aa0/metadata/acquisition_profiles.json) and [docs/API_VERIFICATION.md](https://github.com/virusLuke3/Oracle4SD-Atlas/blob/5ee23c5ee76394233b26c89c1ebd9acae7328aa0/docs/API_VERIFICATION.md). Official [Ethereum JSON-RPC](https://ethereum.org/developers/docs/apis/json-rpc/) and [Polygon endpoint documentation](https://docs.polygon.technology/pos/reference/rpc-endpoints) define the relevant calls. The larger project uses additional chain-native APIs, described in the acquisition profiles. Expected: seven source files, two chains, five transactions and an unchanged data version.

In [ ]:
def read_json(path):
    return json.loads(Path(path).read_text())
cfg = read_json(REPO / "configs/project.json")
spec = read_json(REPO / "data/data_source/uma_case/case_spec.json")
assert cfg["data_version"] == DATA_VERSION
assert len(cfg["sources"]) == 7
for chain, details in spec["chains"].items():
    print(chain, "chain ID", details["chain_id"], "transactions", len(details["transactions"]))
for source in cfg["sources"]:
    data = (REPO / source["path"]).read_bytes()
    assert hashlib.sha256(data).hexdigest() == source["sha256"]
    print(source["source_id"], len(data), "bytes; SHA256 verified")
print("Expected source transactions:", sum(len(c["transactions"]) for c in spec["chains"].values()))

## 3. Querying

For a dependable first run we acquire the frozen public evidence already in the checkout. This **bundled mode** replays actual queried receipts; it makes no new API requests. The next cell invokes [code/query_data/query.py](https://github.com/virusLuke3/Oracle4SD-Atlas/blob/5ee23c5ee76394233b26c89c1ebd9acae7328aa0/code/query_data/query.py) and records every acquired file.

Optional separate runs may use `--mode public-download` (two immutable Hugging Face snapshots, checksum verified) or `--mode live` (five transaction hashes, chain ID, receipts, headers and code). Live mode has bounded retries and may fail when public archive access is unavailable. No pagination is needed for these fixed transaction hashes. Full history requires block ranges or cursor pagination; see the profiles. Expected: mode `bundled`, seven checked files, five source transactions.

In [ ]:
def stage(folder, script, extra=()):
    return command([PYTHON, REPO / "code" / folder / script, "--config", REPO / "configs/project.json", "--run-dir", RUN, *extra], cwd=REPO)
stage("query_data", "query.py", ["--mode", "bundled"])
acquisition = read_json(RUN / "reports/acquisition.json")
assert acquisition["source_transactions"] == 5
assert all(f["frozen_hash_matches"] for f in acquisition["files"])
print({k: acquisition[k] for k in ["mode", "source_transactions", "network_scope", "failure_policy"]})

## 4. Queried-data inspection

Inspect the acquired receipts before deriving observations. A **log** is a contract event emitted in a transaction. A receipt may include logs unrelated to our selected economic flow; we do not count every log as a separate case. Expected: Polygon has four selected receipts and Ethereum one. The hash checks ensure the acquired copies match the registered inputs. Receipt success and exact canonical block matching are checked by the processing/QC stages.

In [ ]:
for chain in ["polygon", "ethereum"]:
    snapshot = read_json(RUN / "queried_data/uma_case/raw" / f"{chain}_rpc_snapshot.json")
    print(chain, {"chain_id": snapshot["chain_id"], "receipts": len(snapshot["receipts"]), "receipt_log_count": sum(len(r["logs"]) for r in snapshot["receipts"].values())})
    assert len(snapshot["receipts"]) == len(spec["chains"][chain]["transactions"])
for entry in acquisition["files"]:
    assert hashlib.sha256((RUN / entry["path"]).read_bytes()).hexdigest() == entry["sha256"]
print("Acquired bytes match the recorded checksums.")

## 5. Processing

The processor decodes event bytes using the **ABI** (the contract interface describing event fields), links a request to its dispute and adjudication, and applies the pinned contract settlement rule. It retains integer token units to avoid rounding and distinguishes returned principal from rewards. One **episode** is the linked lifecycle, not a single transaction.

[code/process_data/process.py](https://github.com/virusLuke3/Oracle4SD-Atlas/blob/5ee23c5ee76394233b26c89c1ebd9acae7328aa0/code/process_data/process.py) calls the existing real-case builder; [docs/METHODS.md](https://github.com/virusLuke3/Oracle4SD-Atlas/blob/5ee23c5ee76394233b26c89c1ebd9acae7328aa0/docs/METHODS.md) explains each transformation. Expected: one episode, nine decoded Polygon events, eight USDC.e transfers and the retained QC/provenance tables. This stage does not infer missing off-chain costs or independent truth.

In [ ]:
stage("process_data", "process.py")
processing = read_json(RUN / "reports/processing.json")
expected_counts = {"economic_episodes": 1, "source_events": 9, "token_flows": 8, "canonical_chain_qc": 30, "economic_link_qc": 13, "variable_provenance": 12, "lifecycle_timeline": 5, "economic_cashflows": 4, "abc_economic_measures": 28}
assert processing["counts"] == expected_counts
print(json.dumps(processing, indent=2))

## 6. Processed-data inspection

The episode has 85 fields; the project's larger common table has a different 49-column schema. The [metadata/data_dictionary.csv](https://github.com/virusLuke3/Oracle4SD-Atlas/blob/5ee23c5ee76394233b26c89c1ebd9acae7328aa0/metadata/data_dictionary.csv) defines every case field, unit, null policy and example. A JSON **null** means unavailable or not identified, not zero. The 28 core A/B/C measures are also distinct from the dictionary's 33 variables, which include five support variables.

The next cell inspects the focal participant and source linkage, lists unavailable fields and confirms that the episode is settled but independent truth is absent. The child tables retain event-level provenance so a reader can trace the result back to receipts.

In [ ]:
episode = read_json(RUN / "processed_data/economic_episode.json")
assert len(episode) == 85
print({key: episode[key] for key in ["episode_id", "actor_role", "terminal_outcome", "right_censored", "ground_truth_status", "source_transaction", "cross_chain_link_grade"]})
print("Unavailable fields:", [key for key, value in episode.items() if value is None])
print("Missingness explanation:", episode["missing_reason"])
with (RUN / "processed_data/lifecycle_timeline.csv").open() as handle:
    timeline = list(csv.DictReader(handle))
print("Lifecycle events:", len(timeline))
display(timeline)

## 7. Analysis

This is a descriptive reconstruction of one settlement. USDC.e uses six decimals, so 1,000,000 raw units equal one token. The winner receives **1,000 USDC.e = 750 returned principal + 250 token gain**. The configured request reward of 5 USDC.e is a separate quantity. Native gas is a different asset and is not subtracted from USDC.e without a timestamped conversion.

The next cell runs [code/analyze_data/analyze.py](https://github.com/virusLuke3/Oracle4SD-Atlas/blob/5ee23c5ee76394233b26c89c1ebd9acae7328aa0/code/analyze_data/analyze.py), displays the accounting components and renders a simple table. It fits no model and supports no population estimate or claim of total net profit.

In [ ]:
stage("analyze_data", "analyze.py")
summary = read_json(RUN / "analysis/summary.json")
scale = 10 ** episode["asset_decimals"]
principal = int(summary["principal_returned_raw"])
gain = int(summary["realized_payoff_raw"])
gross = int(summary["gross_payout_raw"])
assert (principal, gain, gross) == (750_000_000, 250_000_000, 1_000_000_000)
assert principal + gain == gross
display(Markdown("| Component | USDC.e |\n|---|---:|\n" + f"| Returned principal | {principal / scale:,.0f} |\n| Token gain | {gain / scale:,.0f} |\n| Gross transfer | {gross / scale:,.0f} |"))
print("Scope:", summary["scope"])

## 8. Technical validation

Validation asks whether these particular bytes, identities, links and monetary rules are internally consistent. [code/technical_validation/validate.py](https://github.com/virusLuke3/Oracle4SD-Atlas/blob/5ee23c5ee76394233b26c89c1ebd9acae7328aa0/code/technical_validation/validate.py) checks source hashes, the 85-field schema, a preserved earlier summary, original raw-derived assertions and child QC. Expected: **28 top-level checks pass, zero fail**, with 30 canonical and 13 economic/link checks retained underneath. Do not add these levels and call the total independent tests.

The next cell executes validation and displays the denominator for each check. A failed check raises an error rather than silently producing a successful-looking report. The frozen summary is a regression reference, not independent external truth.

In [ ]:
stage("technical_validation", "validate.py")
validation = read_json(RUN / "reports/validation.json")
assert validation["passed"] and validation["passed_checks"] == 28 and validation["failed_checks"] == 0
print({key: validation[key] for key in ["data_version", "passed_checks", "failed_checks", "not_evaluated"]})
display(validation["checks"])

### Deterministic repeat and provenance export

The next cell runs all four stages again in a new directory with the same code and inputs and compares the scientific summary. This is a determinism check, not independent human review. It also runs the student's ten negative/integration tests. The final ZIP preserves **both** runs, the setup identity, environment freeze and exact input/output checksums. On Colab, the final cell offers a download; a local run prints the path. Read the validation denominators and limits before reusing results.


In [ ]:
from IPython import get_ipython
SECOND = REPO / "runs/deterministic-repeat"
print(command([PYTHON, REPO / "scripts/check_repository.py"], cwd=REPO))
print(command([PYTHON, REPO / "scripts/run_pipeline.py", "--output", SECOND], cwd=REPO))
assert read_json(RUN / "reconstruction/statistics/summary.json") == read_json(SECOND / "reconstruction/statistics/summary.json")
print(command([PYTHON, "-m", "unittest", "discover", "-s", "tests", "-v"], cwd=REPO))
(RUN / "reports/environment.freeze.txt").write_text(command([PYTHON, "-m", "pip", "freeze"]))
(RUN / "reports/tutorial_identity.json").write_text(json.dumps({"code_commit": CODE_REF, "data_version": DATA_VERSION, "review_bundle": cfg["review_bundle"], "cutoff": cfg["fixed_cutoff"], "runtime_kind": "hosted_colab" if "google.colab" in sys.modules else "jupyter_kernel" if get_ipython() is not None else "fresh_python_process", "scope": "one real case; same-code deterministic repeat; not full candidate"}, indent=2))
PACKAGE = WORK / "evidence-package"
PACKAGE.mkdir()
shutil.copytree(RUN, PACKAGE / "first-run")
shutil.copytree(SECOND, PACKAGE / "repeat-run")
shutil.copy2(WORK / "setup_identity.json", PACKAGE / "setup_identity.json")
manifest = [{"path": str(p.relative_to(PACKAGE)), "bytes": p.stat().st_size, "sha256": hashlib.sha256(p.read_bytes()).hexdigest()} for p in sorted(PACKAGE.rglob("*")) if p.is_file()]
(PACKAGE / "checksums.json").write_text(json.dumps(manifest, indent=2))
export = shutil.make_archive(str(WORK / "oracle-atlas-case-evidence"), "zip", PACKAGE)
print("Evidence ZIP:", export)
print("SUCCESS: one case reconstructed and checked twice; known full-core issues remain open.")
try:
    from google.colab import files
except ImportError:
    print("Local execution: download or copy the ZIP at the printed path.")
else:
    files.download(export)


### Full dataset, resources and unresolved issues

The [metadata/full_data_inventory.csv](https://github.com/virusLuke3/Oracle4SD-Atlas/blob/5ee23c5ee76394233b26c89c1ebd9acae7328aa0/metadata/full_data_inventory.csv) gives exact hashes and row counts for 71 local candidate files (67 curated plus four applications; 15,474,890,321 bytes). The full immutable public archive and DOI are not available. The existing Hugging Face `1.5.0-draft` review subset is **not** the full core. The [docs/FULL_DATA.md](https://github.com/virusLuke3/Oracle4SD-Atlas/blob/5ee23c5ee76394233b26c89c1ebd9acae7328aa0/docs/FULL_DATA.md) instructions explain authorized-copy verification, isolated production staging, historical source registries, per-protocol commands and why current Pyth rolling state cannot reproduce the historical snapshot. Full collection needs archive endpoints, storage beyond the curated 15.5 GB and a production resource budget; it is not run by this notebook.

Known full-core problems remain in [metadata/issue_log.csv](https://github.com/virusLuke3/Oracle4SD-Atlas/blob/5ee23c5ee76394233b26c89c1ebd9acae7328aa0/metadata/issue_log.csv): dispute/slash labels, Tellor challenge-versus-report semantics, missing reward components, common-schema drift and missing timestamps. This case does not clear those issues. The Overleaf project and compiled manuscript PDF remain author-supplied handoff items; see [docs/HANDOFF.md](https://github.com/virusLuke3/Oracle4SD-Atlas/blob/5ee23c5ee76394233b26c89c1ebd9acae7328aa0/docs/HANDOFF.md).

### Troubleshooting

| Problem | Action |
|---|---|
| Setup says Python 3.12 missing | Select a compatible runtime or follow the local Python 3.12 instructions in the README |
| Package download fails | Restore network access and restart Run all; retain exact requirements |
| Source checksum differs | Re-fetch the pinned revision; never change expected hashes to force a pass |
| Output directory already exists | Run the setup cell again to allocate a new workspace |
| Live endpoint times out or cannot serve archive state | Retain the failed run; use a suitable endpoint or explicitly choose the bundled replay |
| Kernel was interrupted | Restart and Run all; do not continue with partial stage output |
| A scientific assertion fails | Read the stage stderr and validation JSON, keep the evidence and open a scoped issue |

### Review checklist

Confirm the code commit, data version and cutoff printed above; inspect the payment decomposition and null fields; verify the final checks; read the full-data limitations. The repository supplies an isolated local fresh-kernel execution record. That does not claim someone tested the Google-hosted Colab UI or independently reviewed the manuscript.